# Ch.0 Demo — Transformer Concepts Visualized

This notebook provides a hands-on walkthrough of key transformer concepts:
- Tokenization strategies
- Attention mechanisms
- Gradient flow improvements
- Encoder vs Decoder architectures
- RAG pipeline overview

All visualizations use simplified implementations for educational clarity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List

# Configure dark theme for all plots
plt.style.use('dark_background')

print("✓ Setup complete")

## 1. Tokenization Comparison

Different models use different tokenization strategies. **BPE (Byte Pair Encoding)** creates subword vocabularies by merging frequent character pairs.

**Key insight:** Larger vocabularies create fewer tokens but require more memory. Smaller vocabularies are more memory-efficient but create longer sequences.

Below we simulate how the same sentence is tokenized differently:

In [ ]:
def simulate_tokenization(text: str, vocab_size: int) -> List[str]:
    """Simulate BPE tokenization with different vocabulary sizes."""
    # Simulate tokenization - larger vocab = fewer, longer tokens
    if vocab_size == 50000:  # Larger vocab (GPT-4 style)
        # More subwords merged → fewer tokens
        return ["Transform", "ers", " revolution", "ized", " natural", " language", " processing"]
    else:  # Smaller vocab (32000 tokens)
        # Less merging → more tokens
        return ["Trans", "form", "ers", " rev", "olution", "ized", " nat", "ural", " lang", "uage", " process", "ing"]

sentence = "Transformers revolutionized natural language processing"

tokens_large = simulate_tokenization(sentence, vocab_size=50000)
tokens_small = simulate_tokenization(sentence, vocab_size=32000)

print(f"Original: {sentence}\n")
print(f"Large Vocab (50K tokens):")
print(f"  Tokens: {tokens_large}")
print(f"  Count: {len(tokens_large)}\n")

print(f"Small Vocab (32K tokens):")
print(f"  Tokens: {tokens_small}")
print(f"  Count: {len(tokens_small)}\n")

# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 4), facecolor='#1a1a2e')
ax.set_facecolor('#1a1a2e')

models = ['Large Vocab\n(50K)', 'Small Vocab\n(32K)']
token_counts = [len(tokens_large), len(tokens_small)]
colors = ['#4a9eff', '#ff6b6b']

bars = ax.bar(models, token_counts, color=colors, alpha=0.8, edgecolor='white', linewidth=1.5)
ax.set_ylabel('Token Count', fontsize=12, fontweight='bold')
ax.set_title('Tokenization Strategy Impact', fontsize=14, fontweight='bold', pad=20)
ax.set_ylim(0, max(token_counts) * 1.2)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)} tokens',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("💡 Insight: Vocabulary size creates a trade-off between sequence length and memory usage.")

## 2. Attention Heat Map

**Attention is a soft lookup mechanism** that computes similarity between queries (Q) and keys (K), then uses those scores to weight values (V).

The attention formula:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Below we visualize the attention weights (the softmax scores) for a simple 8-token sequence:

In [ ]:
# 8-token sequence
tokens = ["The", "cat", "sat", "on", "the", "mat", "and", "slept"]
seq_len = len(tokens)
d_model = 4  # Embedding dimension (simplified)

# Random embeddings for demonstration
np.random.seed(42)
embeddings = np.random.randn(seq_len, d_model)

# Compute attention scores: Q·Kᵀ
Q = embeddings  # In self-attention, Q = K = V
K = embeddings
scores = Q @ K.T / np.sqrt(d_model)  # Scaled dot-product

# Apply softmax to get attention weights
attention_weights = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)

# Visualize attention heat map
fig, ax = plt.subplots(figsize=(10, 8), facecolor='#1a1a2e')
ax.set_facecolor('#1a1a2e')

im = ax.imshow(attention_weights, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.3)
ax.set_xticks(range(seq_len))
ax.set_yticks(range(seq_len))
ax.set_xticklabels(tokens, fontsize=11, fontweight='bold')
ax.set_yticklabels(tokens, fontsize=11, fontweight='bold')
ax.set_xlabel('Keys (attending to)', fontsize=12, fontweight='bold')
ax.set_ylabel('Queries (attending from)', fontsize=12, fontweight='bold')
ax.set_title('Self-Attention Weights (Q·Kᵀ)', fontsize=14, fontweight='bold', pad=20)

# Add colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Attention Weight', fontsize=11, fontweight='bold')

# Add grid
ax.set_xticks(np.arange(seq_len) - 0.5, minor=True)
ax.set_yticks(np.arange(seq_len) - 0.5, minor=True)
ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.5, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Insight: Each row sums to 1.0 (softmax property). Brighter cells = stronger attention.")
print(f"   Row 1 ('The') attends most to: {tokens[attention_weights[0].argmax()]}")

### Attention in Plain English — How "bank" Understands Context

**What you're about to see:** The word "bank" figures out whether it means "river bank" or "financial bank" by looking at surrounding words.

**No math required** — just watch which words "bank" pays attention to.

In [ ]:
# ===============================================
# Attention Demo: How "bank" Learns Context
# ===============================================

# Sentence with ambiguous word
sentence = ["The", "river", "bank", "was", "flooded", "after", "the", "storm"]
target_word = "bank"  # Which meaning? River or financial?

# Simulate attention scores (in reality, these come from Q@K.T)
# These represent how much "bank" pays attention to each word
attention_scores = {
    "The": 0.05,      # Low attention (just a determiner)
    "river": 0.35,    # HIGH attention (strong geographic clue)
    "bank": 0.12,     # Self-attention (moderate)
    "was": 0.03,      # Low (just a verb)
    "flooded": 0.30,  # HIGH attention (water-related clue)
    "after": 0.04,    # Low (connector word)
    "the": 0.05,      # Low (determiner)
    "storm": 0.06     # Moderate (weather context)
}

# Visualize as ASCII bar chart
print(f"How '{target_word}' distributes its attention:\n")
print("=" * 50)

max_score = max(attention_scores.values())
for word in sentence:
    score = attention_scores[word]
    percentage = int(score * 100)
    bar_length = int((score / max_score) * 30)
    bar = "█" * bar_length + "░" * (30 - bar_length)

    # Color-code interpretation
    if score > 0.25:
        label = "🔥 HIGH"
    elif score > 0.10:
        label = "→ Moderate"
    else:
        label = "   Low"

    print(f"{word:10s} {bar} {percentage:2d}% {label}")

print("=" * 50)
print()

# Explain the result
print("INTERPRETATION:")
print(f"'{target_word}' attended most to:")
top_words = sorted(attention_scores.items(), key=lambda x: x[1], reverse=True)[:3]
for word, score in top_words:
    print(f"  • {word:10s} ({score*100:.0f}%)")

print()
print("CONCLUSION:")
print(f"Since '{target_word}' focused on 'river' (35%) and 'flooded' (30%),")
print("it learns: 'I'm a RIVER BANK (geographic), not a financial bank!'")
print()
print("This is attention in action — context disambiguation.")

**What just happened?**

1. "bank" looked at all surrounding words
2. Computed relevance: river (35%), flooded (30%), storm (6%), others (<5%)
3. Weighted information: mostly copied meaning from "river" and "flooded"
4. Result: "bank" now knows it's geographic, not financial

**Try changing the sentence:**
Replace `sentence = [...]` with:
- `["I", "went", "to", "the", "bank", "to", "deposit", "money"]`
- Now "bank" attends to "deposit" (35%) and "money" (30%) → learns "financial"!

**Key insight:** Same word, different attention, different meaning. That's why transformers are so powerful.

### Interactive Experiment: Attention Temperature

**YOUR TURN:** The `temperature` parameter controls how sharp or smooth attention distributions become. Modify the value below and observe how the heat map changes:

- **temp = 0.1** — Sharp attention (model focuses on 1-2 tokens)
- **temp = 1.0** — Balanced attention (default)
- **temp = 10.0** — Uniform attention (model attends to everything equally)

Try all three values and observe the pattern.

In [ ]:
# ============================================
# YOUR TURN: Experiment with temperature
# ============================================
# Try changing this value: 0.1, 1.0, 10.0
temperature = 1.0  # CHANGE THIS VALUE

# Recompute attention with temperature
scores_temp = scores / temperature  # scores from previous cell
attention_weights_temp = np.exp(scores_temp) / np.exp(scores_temp).sum(axis=1, keepdims=True)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor='#1a1a2e')
fig.suptitle(f'Attention Temperature Comparison (temp={temperature})',
             fontsize=14, color='white', y=0.98)

# Original (temp=1.0)
im1 = ax1.imshow(attention_weights, cmap='YlOrRd', vmin=0, vmax=attention_weights.max())
ax1.set_title('Original (temp=1.0)', fontsize=12, color='white', pad=10)
ax1.set_xlabel('Key Tokens', fontsize=10, color='white')
ax1.set_ylabel('Query Tokens', fontsize=10, color='white')
ax1.set_xticks(range(8))
ax1.set_yticks(range(8))
ax1.set_xticklabels(tokens, rotation=45, ha='right', fontsize=9)
ax1.set_yticklabels(tokens, fontsize=9)
ax1.tick_params(colors='white')
for spine in ax1.spines.values():
    spine.set_color('white')
plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

# Your temperature
im2 = ax2.imshow(attention_weights_temp, cmap='YlOrRd', vmin=0, vmax=attention_weights.max())
ax2.set_title(f'Your Setting (temp={temperature})', fontsize=12, color='white', pad=10)
ax2.set_xlabel('Key Tokens', fontsize=10, color='white')
ax2.set_ylabel('Query Tokens', fontsize=10, color='white')
ax2.set_xticks(range(8))
ax2.set_yticks(range(8))
ax2.set_xticklabels(tokens, rotation=45, ha='right', fontsize=9)
ax2.set_yticklabels(tokens, fontsize=9)
ax2.tick_params(colors='white')
for spine in ax2.spines.values():
    spine.set_color('white')
plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# Print attention distribution statistics
print(f"\nAttention Statistics (temp={temperature}):")
print(f"  Max attention weight: {attention_weights_temp.max():.3f}")
print(f"  Min attention weight: {attention_weights_temp.min():.3f}")
print(f"  Std deviation: {attention_weights_temp.std():.3f}")
print(f"\nInterpretation:")
if temperature < 0.5:
    print("  → Sharp: Model focuses heavily on specific tokens (good for precise tasks)")
elif temperature > 5.0:
    print("  → Uniform: Model attends broadly (may lose focus)")
else:
    print("  → Balanced: Model distributes attention naturally")

## 3. Gradient Flow Comparison

**Why transformers train faster:** Direct attention paths preserve gradients, avoiding the vanishing gradient problem.

We simulate gradient magnitude through 50 layers for three architectures:
1. **Plain Network**: Each layer multiplies gradient by 0.9 → exponential decay
2. **ResNet**: Skip connections add identity mapping → partial preservation
3. **Transformer**: Direct attention paths → full preservation

This explains why transformers can scale to hundreds of layers effectively.

In [ ]:
num_layers = 50
layers = np.arange(1, num_layers + 1)

# Initialize gradients
grad_plain = np.ones(num_layers)
grad_resnet = np.ones(num_layers)
grad_transformer = np.ones(num_layers)

# Simulate gradient flow
for i in range(1, num_layers):
    grad_plain[i] = grad_plain[i-1] * 0.9  # Multiplicative decay
    grad_resnet[i] = grad_resnet[i-1] * 0.9 + 0.1  # Skip connection preserves gradient
    grad_transformer[i] = 1.0  # Direct attention path (simplified)

# Plot gradient magnitude through layers
fig, ax = plt.subplots(figsize=(12, 6), facecolor='#1a1a2e')
ax.set_facecolor('#1a1a2e')

ax.plot(layers, grad_plain, label='Plain Network', color='#ff6b6b', linewidth=2.5, alpha=0.9)
ax.plot(layers, grad_resnet, label='ResNet (Skip Connections)', color='#4ecdc4', linewidth=2.5, alpha=0.9)
ax.plot(layers, grad_transformer, label='Transformer (Attention)', color='#95e1d3', linewidth=2.5, alpha=0.9, linestyle='--')

ax.set_xlabel('Layer Depth', fontsize=12, fontweight='bold')
ax.set_ylabel('Gradient Magnitude', fontsize=12, fontweight='bold')
ax.set_title('Gradient Flow Through Deep Networks', fontsize=14, fontweight='bold', pad=20)
ax.legend(fontsize=11, framealpha=0.9, loc='upper right')
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_ylim(-0.05, 1.15)

# Annotate final values
ax.text(num_layers, grad_plain[-1], f'{grad_plain[-1]:.3f}',
        ha='left', va='center', fontsize=10, fontweight='bold', color='#ff6b6b')
ax.text(num_layers, grad_resnet[-1], f'{grad_resnet[-1]:.3f}',
        ha='left', va='center', fontsize=10, fontweight='bold', color='#4ecdc4')

plt.tight_layout()
plt.show()

print(f"\n📊 Final gradient magnitudes at layer {num_layers}:")
print(f"   Plain Network: {grad_plain[-1]:.4f} (vanished!)")
print(f"   ResNet: {grad_resnet[-1]:.4f} (preserved)")
print(f"   Transformer: {grad_transformer[-1]:.4f} (fully preserved)")
print("\n💡 Insight: Transformers maintain strong gradients even in very deep architectures.")

## 4. Encoder vs Decoder Masking

**BERT (Encoder)** uses **bidirectional attention**: each token can attend to all tokens in the sequence.

**GPT (Decoder)** uses **causal masking**: each token can only attend to previous tokens (autoregressive generation).

The masking pattern determines what information is available during training and inference:

In [ ]:
seq_len = 6
tokens_example = ["I", "love", "machine", "learning", "models", "!"]

# Create attention matrices
encoder_attention = np.ones((seq_len, seq_len))  # Full matrix (bidirectional)
decoder_attention = np.tril(np.ones((seq_len, seq_len)))  # Lower triangular (causal)

# Visualize side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor='#1a1a2e')

# Encoder (BERT-style)
ax1 = axes[0]
ax1.set_facecolor('#1a1a2e')
im1 = ax1.imshow(encoder_attention, cmap='Greens', aspect='auto', vmin=0, vmax=1)
ax1.set_xticks(range(seq_len))
ax1.set_yticks(range(seq_len))
ax1.set_xticklabels(tokens_example, fontsize=10, fontweight='bold', rotation=45)
ax1.set_yticklabels(tokens_example, fontsize=10, fontweight='bold')
ax1.set_title('Encoder (BERT)\nBidirectional Attention', fontsize=12, fontweight='bold', pad=15)
ax1.set_xlabel('Can attend to', fontsize=11, fontweight='bold')
ax1.set_ylabel('Token', fontsize=11, fontweight='bold')

# Add grid
ax1.set_xticks(np.arange(seq_len) - 0.5, minor=True)
ax1.set_yticks(np.arange(seq_len) - 0.5, minor=True)
ax1.grid(which='minor', color='gray', linestyle='-', linewidth=0.5, alpha=0.3)

# Decoder (GPT-style)
ax2 = axes[1]
ax2.set_facecolor('#1a1a2e')
im2 = ax2.imshow(decoder_attention, cmap='Blues', aspect='auto', vmin=0, vmax=1)
ax2.set_xticks(range(seq_len))
ax2.set_yticks(range(seq_len))
ax2.set_xticklabels(tokens_example, fontsize=10, fontweight='bold', rotation=45)
ax2.set_yticklabels(tokens_example, fontsize=10, fontweight='bold')
ax2.set_title('Decoder (GPT)\nCausal Masking', fontsize=12, fontweight='bold', pad=15)
ax2.set_xlabel('Can attend to', fontsize=11, fontweight='bold')
ax2.set_ylabel('Token', fontsize=11, fontweight='bold')

# Add grid
ax2.set_xticks(np.arange(seq_len) - 0.5, minor=True)
ax2.set_yticks(np.arange(seq_len) - 0.5, minor=True)
ax2.grid(which='minor', color='gray', linestyle='-', linewidth=0.5, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Key Differences:")
print("   • Encoder (BERT): Full matrix → understands context bidirectionally")
print("   • Decoder (GPT): Lower triangle → generates left-to-right (causal)")
print("\nThis masking choice determines the model's training objective and use case.")

## 5. RAG Pipeline Overview

**Retrieval-Augmented Generation (RAG)** combines retrieval and generation to ground LLM responses in external knowledge.

### Pipeline Architecture:

```python
# Step 1: Load and embed documents
docs = load_documents()  # Your knowledge base
embeddings = embed_model(docs)  # Convert to vectors

# Step 2: Vector search for relevant context
query_emb = embed_model(user_query)  # Embed user question
top_k = vector_search(query_emb, embeddings, k=5)  # Find top 5 matches

# Step 3: Assemble prompt with retrieved context
prompt = f"""
Context: {top_k}

Question: {user_query}

Answer based on the context above:
"""

# Step 4: Generate answer
answer = llm.generate(prompt)
```

### Component Breakdown:

1. **Document Loading**: Parse PDFs, Markdown, HTML → text chunks
2. **Embedding Model**: `sentence-transformers`, `text-embedding-ada-002`, or custom encoder
3. **Vector Store**: FAISS, Pinecone, Weaviate, ChromaDB
4. **Retrieval**: Cosine similarity or approximate nearest neighbors (ANN)
5. **LLM Generation**: GPT-4, Claude, Llama, etc.

### When to Use RAG:

✅ **Use RAG when:**
- You need factual grounding (cite sources)
- Knowledge changes frequently (news, docs)
- Domain-specific information not in training data
- You want to control what the model "knows"

❌ **Don't use RAG when:**
- Task is purely creative (story generation)
- Latency is critical (retrieval adds overhead)
- Knowledge fits in context window (just use prompt)

### Advanced Techniques:

- **Hybrid Search**: Combine vector similarity + keyword search (BM25)
- **Re-ranking**: Use cross-encoder to re-score top-k results
- **Multi-hop Retrieval**: Iteratively retrieve for complex questions
- **Metadata Filtering**: Filter by date, author, source before retrieval

**Chapters 7-8 will implement this pipeline end-to-end with LangChain and LlamaIndex.**

---

## Summary

This notebook demonstrated:

1. ✅ **Tokenization** — Vocabulary size impacts sequence length and memory
2. ✅ **Attention** — Soft lookup via Q·Kᵀ similarity
3. ✅ **Gradient Flow** — Why transformers scale to deep architectures
4. ✅ **Masking** — Bidirectional (BERT) vs causal (GPT) attention
5. ✅ **RAG Pipeline** — Combining retrieval + generation for grounded responses

**Next Steps:**
- Explore full transformer implementation in Ch.1
- Build RAG systems in Ch.7-8
- Fine-tune models in Ch.9-10